
# Handling Imbalance Class Problem Demonstration In Python

We will begin by building a baseline Random Forest model to establish initial performance. After that, we will apply all four imbalance-handling techniques — random oversampling, random undersampling, SMOTE, and class-weight balancing — to evaluate how each method improves the model’s ability to handle class imbalance.

## Background: 
The large skin-clinic in the city of “XXX” offers variety of products and services to customers. The marketing campaign was launched for introducing a new product.

The data shows response to the marketing campaign alongwith demographic and transaction data of customers.

The objective is to develop a predictive model which can be implemented for the next campaign

## Dataset Description :

| Column Name     | Description                                                                                  |
|-----------------|----------------------------------------------------------------------------------------------|
| Custid          | Unique Customer Identification Code                                                          |
| Age             | Age Group (1: <32 years, 2: 32–48 years, 3: >48 years)                                      |
| Gender          | Gender of Customer (1: Female, 2: Male)                                                     |
| MS              | Marital Status (1: Not Married, 2: Married)                                                 |
| Response        | Customer Campaign Response (1: Responded, 0: Not Responded)                                 |
| Pre_Month       | Purchase Status in Previous Month (1: Purchased, 2: No Purchase)                            |
| N_Products      | Number of Unique Products Purchased in One Year                                             |
| N_Service       | Number of Unique Services Purchased in One Year                                             |
| BillAmt_1       | Bill Amount (USD) for Purchase Transaction 1                                                |
| BillAmt_2       | Bill Amount (USD) for Purchase Transaction 2                                                |
| BillAmt_3       | Bill Amount (USD) for Purchase Transaction 3                                                |


### **Structure:**
1) Import libraries
2) Import & merge data
3) Convert to categorical
4) Create dummies
5) Train/test split
6) Random Forest baseline
7) Note on imbalance
8) Compare methods: oversample, undersample, SMOTE, class_weight='balanced' (AUC + classification tables)


### # 1. Import libraries


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
import warnings
warnings.filterwarnings('ignore')


### 2. Import data (merge)


In [2]:
purchase_data1 = pd.read_csv('Purchase Data 1.csv')
purchase_data2 = pd.read_csv('Purchase Data 2.csv')
response_data = pd.read_csv('Response Data.csv')

# Merge (as in your base notebook)
masterdata = pd.merge(response_data, purchase_data1, how='left', on='Custid')
masterdata = pd.merge(masterdata, purchase_data2, how='left', on='Custid')

print('Shape:', masterdata.shape)
masterdata.head()


Shape: (6990, 11)


,Custid,Age,Gender,MS,Response,Pre_Month,N_Products,N_Service,BillAmt_1,BillAmt_2,BillAmt_3
0,1,2,1,2,1,2,15,24,12.34,13.26,5.88
1,2,2,2,1,0,1,22,29,18.65,2.12,5.13
2,3,1,1,2,0,1,17,21,7.22,3.31,3.65
3,4,2,1,1,0,2,18,22,6.15,2.95,2.34
4,5,2,2,1,0,1,31,35,20.64,2.67,4.07


### 3. Conversion to categorical types

In [3]:

cat_cols = ['Age','Gender','MS','Pre_Month']
for c in cat_cols:
    if c in masterdata.columns:
        masterdata[c] = masterdata[c].astype('category')

masterdata.dtypes


Custid           int64
Age           category
Gender        category
MS            category
Response         int64
Pre_Month     category
N_Products       int64
N_Service        int64
BillAmt_1      float64
BillAmt_2      float64
BillAmt_3      float64
dtype: object

### 4. Create dummies for categorical variables


In [4]:
categorical_vars = [c for c in ['Age','Gender','MS','Pre_Month'] if c in masterdata.columns]
dummies = pd.get_dummies(masterdata[categorical_vars], drop_first=True)
masterdata = pd.concat([masterdata, dummies], axis=1)
print('After dummies shape:', masterdata.shape)

After dummies shape: (6990, 16)


### 5. Prepare X and y, then train-test split


In [5]:
drop_cols = ['Custid','Response']
X = masterdata.drop(columns=[c for c in drop_cols if c in masterdata.columns])
y = masterdata['Response']

# Keep only numeric predictors for simplicity
X = X.select_dtypes(include=[np.number]).fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
print('Class distribution (train):\n', y_train.value_counts(normalize=True))


Train shape: (4893, 5) Test shape: (2097, 5)
Class distribution (train):
 Response
0    0.867566
1    0.132434
Name: proportion, dtype: float64


### 6. Random Forest (baseline on imbalanced data)


In [6]:
rf_baseline = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_baseline.fit(X_train, y_train)

y_pred = rf_baseline.predict(X_test)
y_prob = rf_baseline.predict_proba(X_test)[:,1]

print('Baseline Random Forest AUC:', round(roc_auc_score(y_test, y_prob),4))
print('\nClassification report (baseline):')
print(classification_report(y_test, y_pred))
print('\nConfusion matrix:')
print(confusion_matrix(y_test, y_pred))


Baseline Random Forest AUC: 0.832

Classification report (baseline):
              precision    recall  f1-score   support

           0       0.88      0.97      0.92      1820
           1       0.41      0.12      0.18       277

    accuracy                           0.86      2097
   macro avg       0.64      0.55      0.55      2097
weighted avg       0.82      0.86      0.83      2097


Confusion matrix:
[[1774   46]
 [ 245   32]]



### 7) Imbalanced data problem

When the target classes are imbalanced, models tend to predict the majority class. We'll compare four strategies:

- Baseline (no change)
- Random Oversampling
- Random Undersampling
- SMOTE
- Random Forest with `class_weight='balanced'` (no data resampling)

We'll compare via ROC AUC and classification tables (precision/recall/F1) on the same test set.


In [7]:
#Compare methods (oversample, undersample, SMOTE, class_weight='balanced')
from collections import OrderedDict

def evaluate_model(clf, X_tr, y_tr, X_te, y_te, tag):
    clf.fit(X_tr, y_tr)
    p = clf.predict(X_te)
    prob = clf.predict_proba(X_te)[:,1]
    auc = roc_auc_score(y_te, prob)
    rep = classification_report(y_te, p, output_dict=True)
    return {'method': tag, 'auc': auc, 'report': rep, 'confusion': confusion_matrix(y_te, p)}

results = []

# Oversampling
ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X_train, y_train)
rf_ros = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
results.append(evaluate_model(rf_ros, X_ros, y_ros, X_test, y_test, 'oversample'))

# Undersampling
rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_train, y_train)
rf_rus = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
results.append(evaluate_model(rf_rus, X_rus, y_rus, X_test, y_test, 'undersample'))

# SMOTE
sm = SMOTE(random_state=42)
X_sm, y_sm = sm.fit_resample(X_train, y_train)
rf_sm = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
results.append(evaluate_model(rf_sm, X_sm, y_sm, X_test, y_test, 'smote'))

# Class weight balanced (no resampling)
rf_cw = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
results.append(evaluate_model(rf_cw, X_train, y_train, X_test, y_test, 'class_weight_balanced'))

# Summarize
summary = pd.DataFrame([{'method':r['method'], 'auc':r['auc']} for r in results]).set_index('method')
print('AUC Summary:\n', summary)
print('\nDetailed reports and confusion matrices:')
for r in results:
    print('\n====', r['method'].upper(), '====')
    print('AUC:', round(r['auc'],4))
    print('Confusion matrix:\n', r['confusion'])
    # readable classification report
    print('Classification report:\n', classification_report(y_test, 
          (rf_ros if r['method']=='oversample' else rf_rus if r['method']=='undersample' else rf_sm if r['method']=='smote' else rf_cw).predict(X_test)))


AUC Summary:
                             auc
method                         
oversample             0.835257
undersample            0.813911
smote                  0.831584
class_weight_balanced  0.835105

Detailed reports and confusion matrices:

==== OVERSAMPLE ====
AUC: 0.8353
Confusion matrix:
 [[1707  113]
 [ 192   85]]
Classification report:
               precision    recall  f1-score   support

           0       0.90      0.94      0.92      1820
           1       0.43      0.31      0.36       277

    accuracy                           0.85      2097
   macro avg       0.66      0.62      0.64      2097
weighted avg       0.84      0.85      0.84      2097


==== UNDERSAMPLE ====
AUC: 0.8139
Confusion matrix:
 [[1246  574]
 [  57  220]]
Classification report:
               precision    recall  f1-score   support

           0       0.96      0.68      0.80      1820
           1       0.28      0.79      0.41       277

    accuracy                           0.70      209


### Inference :

In this case, the performance of the model is not significantly improved by any of the methods used to handle imbalanced class problem.